# w9_ps.ipynb — 真·异步参数服务器(Tier-2,@512)

User (2026-07-21 设计): 冒烟代理定 K → shm 权重发布 → 进程通道收梯度 →
版本切片差补偿。**master** 进程持模型+AdamW+64 深版本环,消费
inbox 梯度文件: g' = g + λ·g⊙g·(W_now − W_pull),clip、step、原子重发
{v, vec};每 16 次应用=1 epoch,ckpt 照纪律写;推满 epochs×16 后写
STOP,顺现有投影/选点管线免费出 traj+zsbest+**真实陈旧度直方图**
(ps_staleness_*.json)。**worker**×K 克隆(K=显存冒烟测得单 worker
峰值 → floor((free−6G)×0.85/peak),再按 CPU 核数封顶): 拉最新权重
(记版本 v)→ bf16 单步 i2ce@512 → 扁平 fp32 梯度原子推送 → 循环。
陈旧度从此是**真实的**: 步长期间 master 走了几步就是几,带自然抖动。
对照: i2ce@512 固定分割(.926/.657)+ as8dc5 模拟五折。AUTO-STOPS。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"

ARM = "wcle_i2ce_icetf"
CAP, EPOCHS, DC_LAMBDA = 512, 2000, 0.5
PS_DIR = "/dev/shm/w9_ps_run"
MAX_K_PER_GPU = 10
os.makedirs(OUT_DIR, exist_ok=True)
print(f"PS run: {ARM}@{CAP} {EPOCHS}ep lambda={DC_LAMBDA}")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Smoke-measure -> K, then launch master + K workers.
import json, os, subprocess, tempfile, threading, time
from pathlib import Path

logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()
PSW = os.path.join(REPO, "Pod", "w9_ps_worker.py")
name = f"w9_{ARM}_psdc{int(round(DC_LAMBDA*10))}_fp"

# ---- 1) VRAM smoke proxy (one real-step measurement, plain arm) ----
meas = Path(tempfile.mkdtemp()) / "peak.txt"
cmd = ["python", "-u", PSW, "--data-dir", DATA_DIR, "--out-dir",
       tempfile.mkdtemp(), "--repo", REPO, "--arm", ARM,
       "--anchor-cap", str(CAP), "--epochs", "1",
       "--full-pool", "--full-pool-path", FULL_POOL_PATH,
       "--measure-vram", str(meas)]
print("measuring per-worker peak ...", flush=True)
subprocess.run(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0]))
peak = int(meas.read_text())
free = J.gpu_free_bytes()[gpus[0]] if hasattr(J, "gpu_free_bytes") else 80 << 30
K_mem = int(((free - (6 << 30)) * 0.85) // peak)
K_cpu = max(2, (os.cpu_count() or 16) - 3)
K = max(1, min(K_mem, K_cpu, MAX_K_PER_GPU * len(gpus)))
print(f"peak={peak/2**30:.2f}GiB free={free/2**30:.0f}GiB -> "
      f"K_mem={K_mem} K_cpu={K_cpu} => K={K} workers", flush=True)

# ---- 2) launch master + workers ----
Path(PS_DIR).mkdir(parents=True, exist_ok=True)
def launch(role, wid, gpu):
    lg = open(logd / f"{name}_{role}{wid}.log", "w")
    cmd = ["python", "-u", PSW, "--data-dir", DATA_DIR, "--out-dir", OUT_DIR,
           "--repo", REPO, "--arm", ARM, "--anchor-cap", str(CAP),
           "--epochs", str(EPOCHS), "--ckpt-every", "50",
           "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--ps-role", role, "--ps-dir", PS_DIR, "--ps-id", str(wid),
           "--dc-lambda", str(DC_LAMBDA)]
    return subprocess.Popen(cmd, stdout=lg, stderr=subprocess.STDOUT,
                            env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpu))

procs = [launch("master", 0, gpus[0])]
time.sleep(90)                      # master loads + publishes v0
for w in range(K):
    procs.append(launch("worker", w, gpus[w % len(gpus)]))
    time.sleep(30)                  # host-RAM boot stagger
print(f"master + {K} workers up; waiting for master ...", flush=True)
rc = procs[0].wait()                # master exits after projection+zsbest
Path(PS_DIR, "STOP").write_text("done")   # belt & braces for workers
for pr in procs[1:]:
    pr.wait()
print(f"master rc={rc}; all workers down", flush=True)


In [ ]:
# Readout: real-PS tower vs sync reference + staleness histogram.
import json
from pathlib import Path
name = f"w9_{ARM}_psdc{int(round(DC_LAMBDA*10))}_fp"
zb = Path(OUT_DIR) / f"zsbest_{name.replace('_fp','')}_fp.json"
for nm, lab in ((name.replace('_fp',''), "REAL-PS async"),
                ("w9_wcle_i2ce_icetf", "i2ce@512 sync reference")):
    p = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    if p.exists():
        d = json.loads(p.read_text())
        print(f"{lab:26s} ep{d['best_ep']:>4} neu {d['nm_neutral']:.3f} "
              f"non {d['nm_noname']:.3f} tag {d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    else:
        print(f"{lab:26s} (pending)")
sp = Path(OUT_DIR) / f"ps_staleness_{name.replace('_fp','')}_fp.json"
if sp.exists():
    h = json.loads(sp.read_text())
    tot = sum(h.values())
    mean = sum(int(k)*c for k, c in h.items())/max(tot,1)
    print(f"staleness: mean {mean:.1f}, dist {dict(list(h.items())[:12])}")


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
